# 03 — Xử lý giá trị thiếu

**Mục tiêu:** so sánh các chiến lược điền khuyết trong cùng điều kiện và chọn chiến lược có lập luận.

**Rubric:** Thuộc phần (a) theo đề; kết quả dùng lại cho các notebook 04–08.

**Đề yêu cầu:**
- [ ] Năm chiến lược: trung bình, trung vị, mode, k-NN với k ∈ {3, 5, 10}, MICE.
- [ ] Tạo thêm 10% giá trị thiếu theo MCAR, tính RMSE điền khuyết.
- [ ] Lập bảng so sánh và chọn chiến lược tốt nhất có lập luận.

**Đầu vào:** development set; kết luận về cơ chế thiếu từ notebook 02.

**Đầu ra:** `results/tabular/tables/imputation_rmse.csv`, mask dùng chung, chiến lược được chọn cho từng nhóm cột.
---
*Quy ước khung:* ô **[Code]** sẽ được thay bằng ô code; ô **[Nhận xét]** chỉ viết sau khi có kết quả thật. Mọi score/p-value phải được **in ra và gán vào biến Python** (yêu cầu của đề).

## 0. Thiết lập

> **[Code]** Import thư viện; tìm gốc repo (`find_project_root`, nên chuyển vào `src/data_utils.py`); đọc `data/raw/home_credit/application_train.csv`; nạp `data/splits/home_credit_split_ids.csv` và **chỉ lấy phần development**. Không dùng phần test trong notebook này. `SEED = 42`.

## 1. Thiết kế thí nghiệm

**Lý thuyết — cần trình bày:**
- Vì sao phải **che thêm** các ô đang có giá trị: chỉ ở đó mới biết đáp án đúng.
- $\text{RMSE}_j = \sqrt{\frac{1}{|M_j|}\sum_{i \in M_j}(x_{ij} - \hat x_{ij})^2}$ tính **theo từng cột** $j$; có thể chuẩn hóa bằng độ lệch chuẩn để so sánh giữa cột (NRMSE).
- Cùng một mask cho mọi phương pháp; các phương pháp là **nhánh song song**, không điền nối tiếp.

> **[Code]** Chọn các cột số dùng để đánh giá; tạo mask 10% MCAR trên các ô quan sát được (seed cố định) và lưu lại; nếu KNN/MICE quá chậm thì lấy mẫu — ghi rõ số dòng, seed, cách lấy mẫu.

> **[Nhận xét]** Ghi rõ phạm vi: những cột nào, bao nhiêu dòng, mask có đại diện cho cơ chế thiếu thật không.

## 2. Trung bình, trung vị, mode

**Lý thuyết — cần trình bày:**
- Công thức và tác động: làm giảm phương sai, tạo đỉnh nhọn tại giá trị điền, phá vỡ tương quan.
- Trung vị bền với ngoại lai hơn trung bình; mode cho biến phân loại/rời rạc.

> **[Code]** Điền bằng `SimpleImputer` cho từng chiến lược (fit trên phần train), tính RMSE trên các ô bị che; đo thời gian.

> **[Nhận xét]** So sánh phân phối trước/sau điền cho vài cột lệch mạnh.

## 3. k-NN imputation (k ∈ {3, 5, 10})

**Lý thuyết — cần trình bày:**
- Ý tưởng: điền bằng trung bình của k láng giềng gần nhất, khoảng cách bỏ qua ô thiếu (nan-euclidean).
- Thang đo ảnh hưởng khoảng cách → cần chuẩn hóa trước, tham số chuẩn hóa chỉ học từ train.

> **[Code]** `KNNImputer` với k = 3, 5, 10 trên dữ liệu đã chuẩn hóa; tính RMSE và thời gian cho từng k.

> **[Nhận xét]** k ảnh hưởng thế nào tới RMSE và thời gian chạy?

## 4. MICE (IterativeImputer)

**Lý thuyết — cần trình bày:**
- Chained equations: lần lượt hồi quy từng cột thiếu theo các cột còn lại, lặp đến hội tụ.
- Một lần chạy `IterativeImputer` là **single imputation**; multiple imputation cần nhiều lần với `sample_posterior=True` và gộp kết quả — ghi rõ đang dùng loại nào.

> **[Code]** `IterativeImputer` (bật `enable_iterative_imputer`), ghi số vòng lặp và estimator; tính RMSE và thời gian.

> **[Nhận xét]** MICE có tận dụng được tương quan giữa các cột (notebook 02) không?

## 5. Thiếu "không áp dụng" và biến phân loại

**Lý thuyết — cần trình bày:**
- `OWN_CAR_AGE` thiếu khi không có xe — không có "đáp án đúng" để điền; thường dùng giá trị riêng + cờ chỉ báo.
- Biến phân loại: mode so với tạo nhóm riêng `"Missing"`; RMSE không có nghĩa với mã nghề nghiệp → dùng tỷ lệ khôi phục đúng.

> **[Code]** Xử lý `OWN_CAR_AGE` theo `FLAG_OWN_CAR`; đánh giá tỷ lệ khôi phục đúng cho vài cột phân loại (ví dụ `OCCUPATION_TYPE`) trên mask.

> **[Nhận xét]** Vì sao nhóm cột này cần cách xử lý khác với cột số thông thường?

## 6. Bảng so sánh

**Lý thuyết — cần trình bày:**
- Tiêu chí chọn: RMSE/NRMSE, thời gian, bảo toàn phân phối; có thể thêm điểm CV của baseline khi thay imputer.

> **[Code]** Gộp kết quả vào biến `imputation_results` (phương pháp × cột × RMSE, thời gian) và lưu CSV; vẽ biểu đồ so sánh.

> **[Nhận xét]** Chọn chiến lược cho từng nhóm cột và giải thích. Không gọi là "tốt nhất" ngoài phạm vi đã thử.

## Tổng kết và lựa chọn

> **[Nhận xét]** Chiến lược điền thiếu được chọn cho từng nhóm cột, sẽ dùng trong pipeline cuối.

## Hạn chế

> **[Nhận xét]** Mask MCAR nhân tạo không phản ánh cơ chế thiếu thật (MAR/MNAR); kết quả trên mẫu (nếu có).